RQ3 Table 1: crash vs. assert test counts by model and context variant

Source: `output/success_cases/stat1_error_type_counts_by_model_context.csv`

Output: `output/success_cases/RQ3-table1.csv`

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

CSV_PATH = Path("output/success_cases/stat1_error_type_counts_by_model_context.csv")
OUT_PATH = Path("output/success_cases/RQ3-table1.csv")

df = pd.read_csv(CSV_PATH)
df.shape

(2944, 8)

In [2]:
MODEL_ORDER = ["GPT4o", "Qwen3-coder", "GPTOSS"]
MODEL_LABELS = {"GPT4o": "GPT-4o", "Qwen3-coder": "Qwen3-coder", "GPTOSS": "GPT-OSS"}
CONTEXT_ORDER = ["Minimal", "Method", "Class"]

rows = []
for model in MODEL_ORDER:
    for context in CONTEXT_ORDER:
        sub = df[(df["model"] == model) & (df["context_variant"] == context)]
        crash_tests = int(sub["crash"].sum())
        assert_tests = int(sub["behavioral"].sum())
        total_tests = crash_tests + assert_tests
        pct_crash = round(100 * crash_tests / total_tests, 1) if total_tests else float("nan")
        test_files = int(sub.shape[0])
        rows.append({
            "model": MODEL_LABELS[model],
            "context": context,
            "crash_tests": crash_tests,
            "assert_tests": assert_tests,
            "pct_crash": pct_crash,
            "test_files": test_files,
        })

table = pd.DataFrame(rows)
table

,model,context,crash_tests,assert_tests,pct_crash,test_files
0,GPT-4o,Minimal,169,2,98.8,168
1,GPT-4o,Method,384,0,100.0,292
2,GPT-4o,Class,422,15,96.6,320
3,Qwen3-coder,Minimal,611,81,88.3,653
4,Qwen3-coder,Method,729,52,93.3,654
5,Qwen3-coder,Class,867,9,99.0,706
6,GPT-OSS,Minimal,9,0,100.0,7
7,GPT-OSS,Method,107,10,91.5,77
8,GPT-OSS,Class,92,7,92.9,67


In [3]:
total_crash = int(df["crash"].sum())
total_assert = int(df["behavioral"].sum())
total_pct_crash = round(100 * total_crash / (total_crash + total_assert), 1)
total_test_files = int(df.shape[0])

total_row = pd.DataFrame([{
    "model": "Total",
    "context": "",
    "crash_tests": total_crash,
    "assert_tests": total_assert,
    "pct_crash": total_pct_crash,
    "test_files": total_test_files,
}])

table = pd.concat([table, total_row], ignore_index=True)
table

,model,context,crash_tests,assert_tests,pct_crash,test_files
0,GPT-4o,Minimal,169,2,98.8,168
1,GPT-4o,Method,384,0,100.0,292
2,GPT-4o,Class,422,15,96.6,320
3,Qwen3-coder,Minimal,611,81,88.3,653
4,Qwen3-coder,Method,729,52,93.3,654
5,Qwen3-coder,Class,867,9,99.0,706
6,GPT-OSS,Minimal,9,0,100.0,7
7,GPT-OSS,Method,107,10,91.5,77
8,GPT-OSS,Class,92,7,92.9,67
9,Total,,3390,176,95.1,2944


In [4]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
table.to_csv(OUT_PATH, index=False)
OUT_PATH

PosixPath('output/success_cases/RQ3-table1.csv')

Table VII: how valid LLM-generated tests fail on the breaking version (test level), root cause

Source: `output/success_cases/oracle_and_error_type_per_testcase_detected.csv` (one row per detected test case; `error_type` in {crash, behavioral}).

Output: `output/success_cases/RQ3-table2.csv`

In [5]:
DETECTED_CSV = Path("output/success_cases/oracle_and_error_type_per_testcase_detected.csv")
detected = pd.read_csv(DETECTED_CSV)
detected = detected[detected["error_type"].isin(["behavioral", "crash"])].copy()
total_detected = len(detected)
total_detected

3566

In [6]:
# Assertion (FAILURE) root cause, from the exception name(s) the LLM detected
beh = detected[detected["error_type"] == "behavioral"].copy()

def classify_assertion(errs):
    errs = str(errs)
    if "AssertionFailedError" in errs:
        return "AssertionFailedError"
    if "AssertionError" in errs:
        return "AssertionError"
    return "no explicit assertion"

beh["root_cause"] = beh["llm_detected_errors"].apply(classify_assertion)
assertion_counts = beh["root_cause"].value_counts()
assertion_counts

root_cause
AssertionFailedError     163
no explicit assertion      8
AssertionError             5
Name: count, dtype: int64

In [7]:
# Crash (ERROR) root cause, from the ground-truth matched breaking-change error(s)
crash = detected[detected["error_type"] == "crash"].copy()

CRASH_LABELS = {
    "NoClassDefFoundError": "NoClassDefFoundError",
    "NoSuchMethodError": "NoSuchMethodError",
    "ClassNotFoundException|NoClassDefFoundError": "ClassNotFoundException",
}
crash["root_cause"] = crash["matched_errors"].map(CRASH_LABELS)
crash_counts = crash["root_cause"].value_counts()
crash_counts

root_cause
NoClassDefFoundError      1441
NoSuchMethodError          984
ClassNotFoundException     965
Name: count, dtype: int64

In [8]:
ASSERTION_ORDER = ["AssertionFailedError", "AssertionError", "no explicit assertion"]
CRASH_ORDER = ["NoClassDefFoundError", "NoSuchMethodError", "ClassNotFoundException"]

def pct(n, d):
    return round(100 * n / d, 1)

rows7 = [
    {"outcome": "Assertion (FAILURE)", "test_cases": len(beh), "pct": pct(len(beh), total_detected)},
]
for label in ASSERTION_ORDER:
    n = int(assertion_counts.get(label, 0))
    rows7.append({"outcome": label, "test_cases": n, "pct": pct(n, total_detected)})

rows7.append({"outcome": "Crash (ERROR)", "test_cases": len(crash), "pct": pct(len(crash), total_detected)})
for label in CRASH_ORDER:
    n = int(crash_counts.get(label, 0))
    rows7.append({"outcome": label, "test_cases": n, "pct": pct(n, len(crash))})

table7 = pd.DataFrame(rows7)
table7

,outcome,test_cases,pct
0,Assertion (FAILURE),176,4.9
1,AssertionFailedError,163,4.6
2,AssertionError,5,0.1
3,no explicit assertion,8,0.2
4,Crash (ERROR),3390,95.1
5,NoClassDefFoundError,1441,42.5
6,NoSuchMethodError,984,29.0
7,ClassNotFoundException,965,28.5


In [9]:
OUT_PATH_2 = Path("output/success_cases/RQ3-table2.csv")
table7.to_csv(OUT_PATH_2, index=False)
OUT_PATH_2

PosixPath('output/success_cases/RQ3-table2.csv')

Table VIII: the 176 assertion detections, grouped by the kind of assertion the model wrote

Source: This uses `oracle_methods_used` from the per-test-case CSV loaded above

Output: `output/success_cases/RQ3-table3.csv`

In [10]:
def oracle_intent(o):
    o = str(o)
    if "assertDoesNotThrow" in o:
        return "Should not throw (e.g. assertDoesNotThrow)"
    if o in ("", "nan"):
        return "No explicit assertion"
    if "fail" in o and "assertNotNull" in o:
        return "Existence probe (assertNotNull/fail)"
    return "Value oracle (e.g. assertTrue on output)"

beh["assertion_kind"] = beh["oracle_methods_used"].apply(oracle_intent)
kind_counts = beh["assertion_kind"].value_counts()
kind_counts

assertion_kind
Should not throw (e.g. assertDoesNotThrow)    136
Value oracle (e.g. assertTrue on output)       25
No explicit assertion                           8
Existence probe (assertNotNull/fail)            7
Name: count, dtype: int64

In [11]:
KIND_ORDER = [
    "Should not throw (e.g. assertDoesNotThrow)",
    "Value oracle (e.g. assertTrue on output)",
    "Existence probe (assertNotNull/fail)",
    "No explicit assertion",
]

total_beh = len(beh)
rows8 = [
    {"assertion_kind": k, "test_cases": int(kind_counts.get(k, 0)), "pct": pct(int(kind_counts.get(k, 0)), total_beh)}
    for k in KIND_ORDER
]
rows8.append({"assertion_kind": "Total", "test_cases": total_beh, "pct": 100.0})

table8 = pd.DataFrame(rows8)
table8

,assertion_kind,test_cases,pct
0,Should not throw (e.g. assertDoesNotThrow),136,77.3
1,Value oracle (e.g. assertTrue on output),25,14.2
2,Existence probe (assertNotNull/fail),7,4.0
3,No explicit assertion,8,4.5
4,Total,176,100.0


In [12]:
OUT_PATH_3 = Path("output/success_cases/RQ3-table3.csv")
table8.to_csv(OUT_PATH_3, index=False)
OUT_PATH_3

PosixPath('output/success_cases/RQ3-table3.csv')